# 10. Repair the WebNLG test evaluation from saved per-sample CSVs

This CPU-friendly notebook repairs the existing `fusion_only` and
`full_dual_output` test outputs without regenerating text.

It diagnoses and fixes:

- 5,713 lexicalization rows being treated as independent examples;
- duplicate triple sets receiving repeated identical predictions;
- 1,779 rows with empty references;
- text metrics being computed before references are unioned;
- the old surface-form-sensitive entity metric.

The repaired unit is one unique triple set with one prediction and the union of
all non-empty references.


## 1. Setup


In [ ]:
!pip -q install pandas numpy nltk rouge-score sacrebleu

import os
import json
import ast
import random
from pathlib import Path

import numpy as np
import pandas as pd
import nltk
from google.colab import drive

drive.mount("/content/drive")


## 2. Configuration


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/kg_llm_project"
PROCESSED_DIR = f"{PROJECT_DIR}/baseline-bart-webnlg/processed"
INPUT_DIR = f"{PROJECT_DIR}/seen_unseen_eval"
OUTPUT_DIR = f"{PROJECT_DIR}/repaired_test_evaluation"

FILES = {
    "fusion_only": f"{INPUT_DIR}/per_sample_fusion_only_test.csv",
    "full_dual_output": f"{INPUT_DIR}/per_sample_full_dual_output_test.csv",
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
for name, path in FILES.items():
    print(name, path, "exists=", os.path.exists(path))


## 3. Load train predicates for the seen/unseen split


In [ ]:
import pickle

data_path = f"{PROCESSED_DIR}/webnlg_processed.pkl"
if not os.path.exists(data_path):
    raise FileNotFoundError(data_path)

with open(data_path, "rb") as f:
    processed_data = pickle.load(f)

def _triple_parts(t):
    if isinstance(t, dict):
        return str(t.get("subject", "")), str(t.get("predicate", "")), str(t.get("object", ""))
    return str(t[0]), str(t[1]), str(t[2])

train_predicates = {
    _triple_parts(triple)[1]
    for example in processed_data["train"]
    for triple in example["triples"]
}
print("Training predicates:", len(train_predicates))


## 4. Grounding metric


In [ ]:
# Reference-free grounding metric used throughout the repaired evaluation.
# It normalizes accents, dates, entity shortening, articles, spacing and
# predicate verbalization before calling a mention hallucinated.

import re
import unicodedata
from difflib import SequenceMatcher

_TRANSLIT = {
    "ø":"o","Ø":"o","æ":"ae","Æ":"ae","œ":"oe","Œ":"oe","ð":"d","Ð":"d",
    "þ":"th","Þ":"th","ł":"l","Ł":"l","ß":"ss","đ":"d","Đ":"d","ħ":"h",
    "ı":"i","İ":"i","ŋ":"ng",
}
_MONTHS = [
    "january","february","march","april","may","june",
    "july","august","september","october","november","december",
]
_DET = re.compile(r"^(the|a|an)\s+")
_STOP = frozenset(
    "the a an and or but if then this that these those it its he she they them "
    "his her their in on at of to by with for from as is are was were be been "
    "being there here also however".split()
)

def _strip_accents(s):
    s = "".join(_TRANSLIT.get(c, c) for c in str(s))
    return "".join(
        c for c in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(c)
    )

def _norm(s):
    s = _strip_accents(str(s)).lower().strip().strip('"').strip("'")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return _DET.sub("", s)

def _word_match(text, surface):
    if not surface:
        return False
    return re.search(
        r"(?<![a-z0-9])" + re.escape(surface) + r"(?![a-z0-9])",
        text,
    ) is not None

def _date_surface_tokens(literal):
    tokens = set()
    raw = _strip_accents(str(literal)).strip().strip('"').strip("'")
    m = re.match(r"^(\d{3,4})-(\d{1,2})-(\d{1,2})$", raw)
    if m:
        year, month, day = map(int, m.groups())
        tokens.add(str(year))
        if 1 <= month <= 12:
            tokens.add(_MONTHS[month - 1])
            tokens.add(_MONTHS[month - 1][:3])
        tokens.update({str(day), str(day).zfill(2)})
        tokens.update({f"{day}{suffix}" for suffix in ("st", "nd", "rd", "th")})
    elif re.match(r"^\d{3,4}$", raw):
        tokens.add(raw)
    return tokens

def _digits(s):
    return re.sub(r"[^0-9]", "", str(s))

def _triple_parts(t):
    if isinstance(t, dict):
        return str(t.get("subject", "")), str(t.get("predicate", "")), str(t.get("object", ""))
    return str(t[0]), str(t[1]), str(t[2])

def _split_camel(s):
    return re.sub(r"([a-z])([A-Z])", r"\1 \2", str(s))

def _build_grounding(triples):
    forms, tokens, numcores = [], set(), set()
    for triple in triples:
        subject, predicate, obj = _triple_parts(triple)
        for value in (subject, obj):
            normalized = _norm(value)
            if normalized:
                forms.append(normalized)
                tokens.update(normalized.split())
            tokens.update(_date_surface_tokens(value))
            digit_core = _digits(value)
            if digit_core:
                numcores.add(digit_core)
        tokens.update(_norm(_split_camel(predicate)).split())
    return forms, tokens, numcores

def _extract_mentions(text):
    mentions = set()
    for match in re.findall(
        r"[A-Z][A-Za-z]*(?:[ -][A-Z][A-Za-z]*)*",
        _strip_accents(text),
    ):
        normalized = _norm(match)
        if len(normalized) < 3:
            continue
        if " " not in normalized and normalized in _STOP:
            continue
        mentions.add(normalized)

    for match in re.findall(r"[A-Za-z0-9]+(?:[./\-][A-Za-z0-9]+)*", str(text)):
        if any(char.isdigit() for char in match):
            mentions.add(_norm(match))
    return {mention for mention in mentions if mention}

def grounding_score(prediction, triples):
    pred_norm = _norm(prediction)
    forms, tokens, numcores = _build_grounding(triples)
    forms_despaced = [form.replace(" ", "") for form in forms]

    found = 0
    for form in forms:
        if _word_match(pred_norm, form) or all(
            _word_match(pred_norm, word) for word in form.split()
        ):
            found += 1
    recall = found / len(forms) if forms else 1.0

    mentions = _extract_mentions(prediction)

    def grounded(mention):
        for form in forms:
            if mention == form or _word_match(form, mention):
                return True
        if all(token in tokens for token in mention.split()):
            return True
        mention_digits = _digits(mention)
        if mention_digits and any(
            mention_digits in core or core in mention_digits for core in numcores
        ):
            return True
        despaced = mention.replace(" ", "")
        if len(despaced) >= 3 and any(
            despaced in form or form in despaced for form in forms_despaced
        ):
            return True
        return False

    supported = {m for m in mentions if grounded(m)}
    hallucinated = mentions - supported
    precision = len(supported) / len(mentions) if mentions else 1.0

    # Diagnostic-only categorization:
    # "in_graph_corruption" means a hallucinated mention is string-similar to
    # an input entity/literal, rather than a wholly external addition.
    corruption = []
    external = []
    for mention in sorted(hallucinated):
        best = max(
            (SequenceMatcher(None, mention, form).ratio() for form in forms),
            default=0.0,
        )
        if best >= 0.55:
            corruption.append(mention)
        else:
            external.append(mention)

    return {
        "entity_precision": precision,
        "entity_recall": recall,
        "hallucination_rate": 1.0 - precision,
        "hallucinated_entities": sorted(hallucinated),
        "in_graph_corruptions": corruption,
        "external_hallucinations": external,
    }


## 5. Diagnose and deduplicate each saved output


In [ ]:
from collections import OrderedDict

def parse_jsonish(value, default):
    if isinstance(value, (list, dict)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return default
    text = str(value).strip()
    if not text:
        return default
    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except Exception:
            pass
    return default

def canonical_triple_key(triples):
    # Preserve order: different triple orders are different linearized model inputs.
    normalized = [_triple_parts(triple) for triple in triples]
    return json.dumps(normalized, ensure_ascii=False)

def row_references(row):
    references = parse_jsonish(row.get("all_references", "[]"), [])
    if not isinstance(references, list):
        references = [references]
    references = [str(value).strip() for value in references if str(value).strip()]
    raw_reference = row.get("reference", "")
    reference = "" if pd.isna(raw_reference) else str(raw_reference).strip()
    if reference and reference.lower() != "nan" and reference not in references:
        references.append(reference)
    return references

def repair_file(path, model_name):
    frame = pd.read_csv(path)
    groups = OrderedDict()

    empty_reference_rows = 0
    for row_index, row in frame.iterrows():
        triples = parse_jsonish(row["input_triples"], [])
        key = canonical_triple_key(triples)
        refs = row_references(row)
        if not refs:
            empty_reference_rows += 1

        if key not in groups:
            groups[key] = {
                "triples": triples,
                "predictions": [],
                "references": [],
                "row_indices": [],
            }
        group = groups[key]
        prediction = str(row.get("prediction", "") or "")
        if prediction not in group["predictions"]:
            group["predictions"].append(prediction)
        for reference in refs:
            if reference not in group["references"]:
                group["references"].append(reference)
        group["row_indices"].append(int(row_index))

    repaired = []
    inconsistent_prediction_groups = 0
    for unique_id, (key, group) in enumerate(groups.items()):
        if len(group["predictions"]) != 1:
            inconsistent_prediction_groups += 1
        if not group["references"]:
            raise ValueError(f"{model_name} unique group {unique_id} has no reference.")

        predicates = sorted({_triple_parts(t)[1] for t in group["triples"]})
        novel = sorted(set(predicates) - train_predicates)
        repaired.append({
            "unique_id": unique_id,
            "key": key,
            "input_triples": group["triples"],
            "prediction": group["predictions"][0],
            "references": group["references"],
            "original_row_indices": group["row_indices"],
            "n_original_rows": len(group["row_indices"]),
            "n_distinct_predictions": len(group["predictions"]),
            "predicates": predicates,
            "novel_predicates": novel,
            "split": "unseen" if novel else "seen",
        })

    diagnostics = {
        "model": model_name,
        "raw_rows": len(frame),
        "unique_triple_sets": len(repaired),
        "duplicate_extra_rows": len(frame) - len(repaired),
        "duplicate_extra_fraction": (len(frame) - len(repaired)) / len(frame),
        "empty_reference_rows": empty_reference_rows,
        "empty_reference_fraction": empty_reference_rows / len(frame),
        "inconsistent_prediction_groups": inconsistent_prediction_groups,
    }
    return repaired, diagnostics

repaired_by_model = {}
diagnostics = []
for model_name, path in FILES.items():
    repaired, report = repair_file(path, model_name)
    repaired_by_model[model_name] = repaired
    diagnostics.append(report)

diagnostic_frame = pd.DataFrame(diagnostics)
display(diagnostic_frame)
diagnostic_frame.to_csv(
    os.path.join(OUTPUT_DIR, "input_diagnostics.csv"),
    index=False,
)


## 6. Correct text and grounding metrics


In [ ]:
# Correct text and grounding metrics for variable-reference examples.

import math
import statistics
import numpy as np
import pandas as pd
import nltk
import sacrebleu
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from rouge_score import rouge_scorer

for package in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    try:
        nltk.download(package, quiet=True)
    except Exception:
        pass

_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def score_predictions(predictions, examples, metas):
    if len(predictions) != len(examples):
        raise ValueError("Prediction/example length mismatch.")

    references_list = [example["all_targets"] for example in examples]
    prediction_tokens = [prediction.split() for prediction in predictions]
    reference_tokens = [
        [reference.split() for reference in references]
        for references in references_list
    ]

    nltk_bleu = corpus_bleu(
        reference_tokens,
        prediction_tokens,
        smoothing_function=SmoothingFunction().method1,
    ) * 100.0

    max_references = max(len(refs) for refs in references_list)
    reference_streams = []
    for reference_index in range(max_references):
        stream = []
        for refs in references_list:
            stream.append(refs[reference_index] if reference_index < len(refs) else refs[0])
        reference_streams.append(stream)

    sacre = sacrebleu.corpus_bleu(
        predictions,
        reference_streams,
        lowercase=True,
        tokenize="13a",
    ).score

    meteor_values = []
    rouge_values = []
    grounding_rows = []
    for prediction, example, meta in zip(predictions, examples, metas):
        references = example["all_targets"]
        pred_tokens = prediction.split()
        ref_tokens = [reference.split() for reference in references]
        try:
            meteor = float(nltk_meteor(ref_tokens, pred_tokens))
        except Exception:
            meteor = 0.0
        rouge = max(
            _rouge.score(reference, prediction)["rougeL"].fmeasure
            for reference in references
        )
        grounding = grounding_score(prediction, example["triples"])
        meteor_values.append(meteor)
        rouge_values.append(rouge)
        grounding_rows.append({
            **meta,
            "prediction": prediction,
            "references": references,
            **grounding,
        })

    frame = pd.DataFrame(grounding_rows)
    summary = {
        "n": len(predictions),
        "nltk_corpus_bleu": float(nltk_bleu),
        "sacrebleu_13a_lower": float(sacre),
        "meteor": float(np.mean(meteor_values)),
        "rouge_l": float(np.mean(rouge_values)),
        "entity_precision": float(frame["entity_precision"].mean()),
        "entity_recall": float(frame["entity_recall"].mean()),
        "hallucination_rate": float(frame["hallucination_rate"].mean()),
        "examples_with_hallucination": float(
            (frame["hallucination_rate"] > 0).mean()
        ),
        "in_graph_corruption_mentions": int(
            frame["in_graph_corruptions"].map(len).sum()
        ),
        "external_hallucination_mentions": int(
            frame["external_hallucinations"].map(len).sum()
        ),
    }
    return summary, frame

def score_by_split(predictions, examples, metas):
    outputs = {}
    for split_name in ("overall", "seen", "unseen"):
        if split_name == "overall":
            indices = list(range(len(examples)))
        else:
            indices = [
                index for index, meta in enumerate(metas)
                if meta["split"] == split_name
            ]
        if not indices:
            print(f"Skipping empty split: {split_name}")
            continue
        split_predictions = [predictions[index] for index in indices]
        split_examples = [examples[index] for index in indices]
        split_metas = [metas[index] for index in indices]
        summary, frame = score_predictions(
            split_predictions,
            split_examples,
            split_metas,
        )
        outputs[split_name] = (summary, frame)
    return outputs

def alpha_tag(alpha):
    return str(alpha).replace(".", "p").replace("-", "m")

def export_official_text_files(tag, predictions, examples):
    out_dir = os.path.join(OUTPUT_DIR, "official_text_files", tag)
    os.makedirs(out_dir, exist_ok=True)

    candidate_path = os.path.join(out_dir, "candidate.txt")
    with open(candidate_path, "w", encoding="utf-8") as f:
        for prediction in predictions:
            f.write(prediction.replace("\n", " ").strip() + "\n")

    max_references = max(len(example["all_targets"]) for example in examples)
    reference_paths = []
    for reference_index in range(max_references):
        path = os.path.join(out_dir, f"reference{reference_index}.txt")
        reference_paths.append(path)
        with open(path, "w", encoding="utf-8") as f:
            for example in examples:
                refs = example["all_targets"]
                reference = refs[reference_index] if reference_index < len(refs) else refs[0]
                f.write(reference.replace("\n", " ").strip() + "\n")
    return candidate_path, reference_paths


In [ ]:
summaries = []
scored_frames = {}

for model_name, rows in repaired_by_model.items():
    predictions = [row["prediction"] for row in rows]
    examples = [
        {
            "triples": row["input_triples"],
            "all_targets": row["references"],
            "target": row["references"][0],
        }
        for row in rows
    ]
    metas = [
        {
            key: value
            for key, value in row.items()
            if key not in {"prediction", "references", "input_triples"}
        }
        for row in rows
    ]

    split_outputs = score_by_split(predictions, examples, metas)
    scored_frames[model_name] = split_outputs["overall"][1]

    for split_name, (summary, scored) in split_outputs.items():
        summaries.append({
            "model": model_name,
            "split": split_name,
            **summary,
        })
        export = scored.copy()
        for column in export.columns:
            if export[column].map(lambda x: isinstance(x, (list, dict))).any():
                export[column] = export[column].map(
                    lambda x: json.dumps(x, ensure_ascii=False)
                    if isinstance(x, (list, dict)) else x
                )
        export.to_csv(
            os.path.join(OUTPUT_DIR, f"per_sample_{model_name}_{split_name}.csv"),
            index=False,
        )

    export_official_text_files(model_name, predictions, examples)

summary_frame = pd.DataFrame(summaries)
summary_frame.to_csv(
    os.path.join(OUTPUT_DIR, "repaired_metrics_summary.csv"),
    index=False,
)
display(summary_frame)


## 7. Paired fusion-only versus full-dual analysis


In [ ]:
fusion = scored_frames["fusion_only"].set_index("key")
dual = scored_frames["full_dual_output"].set_index("key")
common_keys = fusion.index.intersection(dual.index)

delta = (
    dual.loc[common_keys, "hallucination_rate"].to_numpy()
    - fusion.loc[common_keys, "hallucination_rate"].to_numpy()
)

rng = np.random.default_rng(42)
boot = []
for _ in range(5000):
    indices = rng.integers(0, len(delta), size=len(delta))
    boot.append(float(delta[indices].mean()))

paired = pd.DataFrame([{
    "n_paired": len(delta),
    "full_minus_fusion_hallucination_pp": float(delta.mean() * 100),
    "ci95_low_pp": float(np.quantile(boot, 0.025) * 100),
    "ci95_high_pp": float(np.quantile(boot, 0.975) * 100),
    "full_better_examples": int((delta < 0).sum()),
    "full_worse_examples": int((delta > 0).sum()),
    "same_examples": int((delta == 0).sum()),
}])

paired.to_csv(
    os.path.join(OUTPUT_DIR, "paired_fusion_vs_full.csv"),
    index=False,
)
display(paired)
print("All repaired outputs saved to:", OUTPUT_DIR)
